# SIAM Conference on Computational Science and Engineering
`CSE19` https://meetings.siam.org/program.cfm?CONFCODE=CS19<br>
`CSE23` https://meetings.siam.org/program.cfm?CONFCODE=cse23<br>
`CSE25` https://meetings.siam.org/program.cfm?CONFCODE=cse25<br>

# Initialize

In [1]:
# ==== Step 0. Import libraries ====
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import numpy as np
import random
import plotly.express as px

In [2]:
# ==== Step 0. Load JSON file ====
with open('CSE23_schd.json') as f:
    data = json.load(f)

conference_year = "2023"

In [3]:
# ==== Step 1. Conference Programme Timeline Coloured by Popularity Score ====
def prepareTimelineCitations3(data, map, modelName = "", fileName = "schedule.html", useLogScale = False):
    # Flatten into a DataFrame
    events = list(data.values())

    # Keep only those with codes starting with 'MS'
    events = [e for e in events if e.get("code", "").startswith("MS")]

    # Clean missing data
    for e in events:
        if map.get(e["code"]) is not None:
            e["total_recent_cite_count"] = (map[e["code"]])
        else:
            e["total_recent_cite_count"] = -1
        if pd.isna(e.get("room", None)):
            e["room"] = "NA"

    df = pd.DataFrame(events)

    # Parse datetimes (2025 as the year)
    df['start'] = pd.to_datetime(df['day'] + ' ' + df['begin_time'] + f" {conference_year}",
                                format='%A, %B %d %I:%M %p %Y')
    df['end']   = pd.to_datetime(df['day'] + ' ' + df['end_time']   + f" {conference_year}",
                                format='%A, %B %d %I:%M %p %Y')

    # Preserve human-readable start/end for tooltip
    df['start_str'] = df['start'].dt.strftime('%b %d %H:%M')
    df['end_str'] = df['end'].dt.strftime('%b %d %H:%M')

    # Original citation metric kept for tooltip
    df['cite_orig'] = df['total_recent_cite_count'].astype(float)

    if useLogScale:
        # apply log to the original but keep an unclipped version for display
        df['cite_for_scale'] = np.log1p(df['cite_orig'])
    else:
        df['cite_for_scale'] = df['cite_orig']

    # Cap scale at 20,000 (or log(1+20000) if log scale)
    if useLogScale:
        cap_value = np.log1p(20000)
    else:
        cap_value = 20000.0
    df['cite_plot'] = np.minimum(df['cite_for_scale'], cap_value)

    # Create the timeline; put code/start/end and original cite in custom_data for tooltip
    fig = px.timeline(
        df,
        x_start="start",
        x_end="end",
        y="room",
        color="cite_plot",
        category_orders={"room": sorted(df['room'].unique())},
        title=f"Conference Schedule {modelName}",
        custom_data=["code", "start_str", "end_str", "cite_orig"]
    )
    fig.update_yaxes(autorange="reversed")

    # Force colorbar range so anything above cap shows as max
    fig.update_traces(
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Room: %{y}<br>"
            "Start: %{customdata[1]}<br>"
            "End: %{customdata[2]}<br>"
            "Popularity Score: %{customdata[3]:.2f}<extra></extra>"
        )
    )
    fig.update_layout(
        coloraxis_colorbar=dict(
            title="Popularity Score",
        )
    )
    # Since px.timeline used a mapped discrete color axis, adjust via update to ensure continuous scale
    fig.update_traces(marker=dict(
        colorbar=dict(title="Popularity Score"),
        cmin=0,
        cmax=cap_value,
        showscale=True
    ))

    # Set the x-axis range to start at the beginning of the first day and format ticks
    min_date = df['start'].min().normalize()
    max_date = df['end'].max().normalize()
    fig.update_layout(
        xaxis=dict(
            range=[min_date, max_date],
            tickformat='%b %d',  # Format ticks to show month and day
            dtick='d'            # Place ticks at the start of each day
        )
    )

    # Add day separators
    shapes = []
    for day in sorted(df['start'].dt.normalize().unique()):
        sep = day + pd.Timedelta(days=1)
        shapes.append({
            "type": "line",
            "x0": sep, "x1": sep,
            "y0": -0.5, "y1": len(df['room'].unique()) - 0.5,
            "line": {"color": "gray", "width": 1, "dash": "dash"}
        })
    fig.update_layout(shapes=shapes)

    # Export to standalone HTML
    fig.write_html(fileName, full_html=True)
    print(f"→ {fileName} written with code only in tooltip and capped color scale.")

## Citation Visualization

## Load from file (Map MS: CitationCount)

In [4]:
map_open_alex_all = {'IP1': 0, 'MS1': 9298.0, 'MS2': 6627.0, 'MS3': 1230.0, 'MS4': 89.0, 'MS5': 363.0, 'MS6': 3113.0, 'MS7': 1295.0, 'MS10': 5517.0, 'MS11': 9631.0, 'MS12': 2318.0, 'MS13': 12086.0, 'MS14': 12528.0, 'MS15': 8154.0, 'MS16': 2462.0, 'MS17': 179.0, 'MS18': 2861.0, 'MS19': 7057.0, 'MS20': 3840.0, 'MS21': 2732.0, 'MS22': 11216.0, 'MS23': 4226.0, 'MS24': 786.0, 'MS25': 3053.0, 'MS26': 14105.0, 'MS27': 1270.0, 'MS28': 15666.0, 'MS29': 5735.0, 'MS30': 7526.0, 'MS31': 4470.0, 'MS32': 1272.0, 'MS33': 3200.0, 'MS34': 13983.0, 'MS35': 7065.0, 'MS84': 3036.0, 'MS373': 11849.0, 'IP2': 0, 'MS36': 656.0, 'MS37': 266.0, 'MS38': 4744.0, 'MS39': 9295.0, 'MS40': 1324.0, 'MS42': 572.0, 'MS43': 9972.0, 'MS44': 11458.0, 'MS45': 2152.0, 'MS46': 25648.0, 'MS47': 2115.0, 'MS48': 10610.0, 'MS49': 0, 'MS50': 1926.0, 'MS51': 3804.0, 'MS52': 58833.0, 'MS53': 43111.0, 'MS54': 3329.0, 'MS55': 8609.0, 'MS56': 7533.0, 'MS57': 0, 'MS58': 409.0, 'MS59': 7962.0, 'MS60': 6252.0, 'MS61': 2212.0, 'MS62': 14590.0, 'MS63': 77201.0, 'MS64': 30910.0, 'MS65': 28871.0, 'MS66': 23817.0, 'MS67': 0, 'MS68': 2012.0, 'MS69': 6929.0, 'MS70': 1597.0, 'MS121': 4687.0, 'SP1': 0, 'SP2': 0, 'SP3': 0, 'SP4': 0, 'IP3': 0, 'MS9': 5828.0, 'MS71': 6937.0, 'MS72': 8585.0, 'MS73': 11059.0, 'MS74': 16304.0, 'MS75': 11633.0, 'MS76': 13772.0, 'MS77': 2137.0, 'MS78': 3688.0, 'MS79': 8603.0, 'MS80': 6539.0, 'MS81': 5450.0, 'MS82': 14182.0, 'MS83': 15100.0, 'MS85': 13356.0, 'MS86': 1518.0, 'MS87': 8425.0, 'MS88': 22556.0, 'MS89': 6246.0, 'MS90': 21179.0, 'MS91': 22535.0, 'MS92': 5699.0, 'MS93': 17192.0, 'MS94': 22712.0, 'MS95': 6523.0, 'MS96': 14009.0, 'MS97': 16557.0, 'MS98': 19071.0, 'MS99': 4837.0, 'MS100': 10655.0, 'MS101': 6308.0, 'MS102': 17378.0, 'MS103': 905.0, 'MS104': 3023.0, 'MS105': 7826.0, 'MS106': 11348.0, 'PD1': 0, 'IP4': 0, 'MS41': 6663.0, 'MS107': 2389.0, 'MS108': 4212.0, 'MS109': 703.0, 'MS110': 22628.0, 'MS111': 10390.0, 'MS112': 8579.0, 'MS113': 1310.0, 'MS114': 1999.0, 'MS115': 395.0, 'MS116': 2763.0, 'MS117': 4009.0, 'MS118': 2288.0, 'MS119': 7701.0, 'MS120': 2367.0, 'MS122': 1520.0, 'MS123': 5624.0, 'MS124': 17106.0, 'MS125': 5511.0, 'MS126': 1773.0, 'MS127': 11964.0, 'MS128': 1812.0, 'MS129': 7634.0, 'MS130': 9847.0, 'MS131': 7863.0, 'MS132': 10612.0, 'MS133': 4833.0, 'MS134': 228.0, 'MS135': 9981.0, 'MS136': 4829.0, 'MS137': 666.0, 'MS138': 4457.0, 'MS139': 14482.0, 'MS140': 832.0, 'MS141': 4311.0, 'MS142': 1226.0, 'IP5': 0, 'MS143': 12666.0, 'MS144': 9472.0, 'MS145': 242.0, 'MS146': 5310.0, 'MS147': 2747.0, 'MS148': 2520.0, 'MS149': 12.0, 'MS150': 2843.0, 'MS151': 3703.0, 'MS152': 2842.0, 'MS153': 6190.0, 'MS154': 806.0, 'MS155': 3052.0, 'MS156': 2266.0, 'MS157': 17320.0, 'MS158': 19196.0, 'MS159': 328.0, 'MS160': 3303.0, 'MS161': 3819.0, 'MS162': 13225.0, 'MS163': 10700.0, 'MS164': 7161.0, 'MS165': 19738.0, 'MS166': 3656.0, 'MS167': 3568.0, 'MS168': 4933.0, 'MS169': 6267.0, 'MS170': 12515.0, 'MS171': 15161.0, 'MS172': 799.0, 'MS173': 19487.0, 'MS174': 7516.0, 'MS175': 4770.0, 'MS176': 11391.0, 'MS177': 721.0, 'PD2': 0, 'IP6': 0, 'MS178': 2391.0, 'MS179': 6214.0, 'MS180': 347.0, 'MS181': 13572.0, 'MS182': 6865.0, 'MS183': 4131.0, 'MS184': 3552.0, 'MS185': 4210.0, 'MS186': 15477.0, 'MS187': 1054.0, 'MS188': 6517.0, 'MS189': 4953.0, 'MS190': 9683.0, 'MS191': 22290.0, 'MS192': 7828.0, 'MS193': 9223.0, 'MS194': 2175.0, 'MS195': 4173.0, 'MS196': 5394.0, 'MS197': 2124.0, 'MS198': 2348.0, 'MS199': 4634.0, 'MS200': 3553.0, 'MS201': 8230.0, 'MS202': 11631.0, 'MS203': 3608.0, 'MS204': 3293.0, 'MS205': 5142.0, 'MS206': 50764.0, 'MS207': 6363.0, 'MS208': 6265.0, 'MS209': 3902.0, 'MS210': 612.0, 'MS211': 2988.0, 'MS212': 8042.0, 'MS213': 5866.0, 'MS214': 3531.0, 'MS215': 2445.0, 'MS216': 5704.0, 'MS217': 365.0, 'MS218': 5208.0, 'MS219': 30457.0, 'MS220': 50773.0, 'MS221': 10694.0, 'MS222': 45693.0, 'MS223': 21964.0, 'MS224': 15752.0, 'MS225': 9228.0, 'MS226': 2192.0, 'MS227': 3345.0, 'MS228': 9512.0, 'MS229': 4868.0, 'MS230': 9759.0, 'MS231': 1244.0, 'MS232': 9565.0, 'MS233': 17166.0, 'MS234': 1092.0, 'MS235': 1087.0, 'MS236': 4068.0, 'MS237': 22772.0, 'MS238': 385.0, 'MS239': 0, 'MS240': 7127.0, 'MS241': 11625.0, 'MS242': 2055.0, 'MS243': 7181.0, 'MS244': 4317.0, 'MS245': 6879.0, 'MS246': 1533.0, 'MS247': 4872.0, 'MS8': 5031.0, 'MS450': 5580.0, 'IP7': 0, 'MS248': 21323.0, 'MS249': 40164.0, 'MS250': 7051.0, 'MS251': 9247.0, 'MS252': 4716.0, 'MS253': 3170.0, 'MS254': 6491.0, 'MS255': 10860.0, 'MS256': 6114.0, 'MS257': 29331.0, 'MS258': 3211.0, 'MS259': 12060.0, 'MS260': 3315.0, 'MS261': 24913.0, 'MS262': 3189.0, 'MS263': 8423.0, 'MS264': 36212.0, 'MS265': 1735.0, 'MS266': 15113.0, 'MS267': 6821.0, 'MS268': 22264.0, 'MS269': 5139.0, 'MS270': 3666.0, 'MS271': 11872.0, 'MS272': 4899.0, 'MS273': 18205.0, 'MS274': 23091.0, 'MS275': 46.0, 'MS276': 966.0, 'MS277': 3229.0, 'MS278': 530.0, 'MS279': 3583.0, 'MS280': 3901.0, 'MS281': 4292.0, 'MS282': 3166.0, 'PD3': 0, 'PD4': 0, 'MS283': 9175.0, 'MS284': 21383.0, 'MS285': 15024.0, 'MS286': 25435.0, 'MS287': 4617.0, 'MS288': 2341.0, 'MS289': 2460.0, 'MS290': 4053.0, 'MS291': 5651.0, 'MS292': 15911.0, 'MS293': 8265.0, 'MS294': 6596.0, 'MS295': 3373.0, 'MS296': 14799.0, 'MS297': 6053.0, 'MS298': 8870.0, 'MS299': 15790.0, 'MS300': 114.0, 'MS301': 18213.0, 'MS302': 1190.0, 'MS303': 4567.0, 'MS304': 4965.0, 'MS305': 4187.0, 'MS306': 3030.0, 'MS307': 1134.0, 'MS308': 6542.0, 'MS309': 4165.0, 'MS310': 634.0, 'MS311': 6624.0, 'MS312': 12446.0, 'MS313': 27518.0, 'MS314': 80.0, 'MS315': 6123.0, 'MS316': 7553.0, 'MS317': 17328.0, 'MS318': 0, 'MS319': 0, 'MS320': 6581.0, 'MS321': 6656.0, 'MS322': 4863.0, 'MS323': 7102.0, 'MS324': 12742.0, 'MS325': 1718.0, 'MS326': 11078.0, 'MS327': 8311.0, 'MS328': 994.0, 'MS329': 6604.0, 'MS330': 10311.0, 'MS331': 0, 'MS332': 4781.0, 'MS333': 4668.0, 'MS334': 0, 'MS335': 5052.0, 'MS336': 2193.0, 'MS337': 8922.0, 'MS338': 253.0, 'MS339': 836.0, 'MS340': 12128.0, 'MS341': 9038.0, 'MS342': 6109.0, 'MS343': 3051.0, 'MS344': 12481.0, 'MS345': 20809.0, 'MS346': 0, 'MS347': 953.0, 'MS348': 597.0, 'MS349': 28207.0, 'MS350': 15334.0, 'MS351': 30034.0, 'MS352': 5300.0, 'IP8': 0, 'MS353': 33395.0, 'MS354': 30067.0, 'MS355': 1346.0, 'MS356': 5570.0, 'MS357': 14940.0, 'MS358': 2752.0, 'MS359': 3187.0, 'MS360': 7264.0, 'MS361': 19491.0, 'MS362': 1083.0, 'MS363': 930.0, 'MS364': 1684.0, 'MS365': 28.0, 'MS366': 7289.0, 'MS367': 10582.0, 'MS368': 2307.0, 'MS369': 5722.0, 'MS370': 19304.0, 'MS371': 14757.0, 'MS372': 6824.0, 'MS374': 270.0, 'MS375': 8724.0, 'MS376': 3013.0, 'MS377': 54185.0, 'MS378': 122601.0, 'MS379': 5579.0, 'MS380': 11451.0, 'MS381': 53093.0, 'MS382': 60060.0, 'MS383': 5196.0, 'MS384': 4171.0, 'MS385': 1534.0, 'MS386': 5116.0, 'MS387': 11.0, 'MS388': 8043.0, 'MS389': 14121.0, 'MS390': 2817.0, 'MS391': 2834.0, 'MS392': 9585.0, 'MS393': 20081.0, 'MS394': 15002.0, 'MS395': 6938.0, 'MS396': 1393.0, 'MS397': 10502.0, 'MS398': 25862.0, 'MS399': 3777.0, 'MS400': 7794.0, 'MS401': 34067.0, 'MS402': 14794.0, 'MS403': 8374.0, 'MS404': 8046.0, 'MS405': 2609.0, 'MS406': 33205.0, 'MS407': 12837.0, 'MS408': 15855.0, 'MS409': 3820.0, 'MS410': 4878.0, 'MS411': 5474.0, 'MS412': 25106.0, 'MS413': 3269.0, 'MS414': 18349.0, 'MS415': 1591.0, 'MS416': 25488.0, 'MS417': 11149.0, 'MS418': 5065.0, 'MS419': 1663.0, 'MS420': 249.0, 'MS421': 21310.0, 'MS422': 2835.0}

In [5]:
with open("map_ms_citation.json", "w") as f:
    json.dump(map_open_alex_all, f)

In [6]:
# MAP_FILE = "CSE_2023_MS_CitationMap.json"
MAP_FILE = "map_ms_citation.json"

with(open(MAP_FILE, "r")) as f:
    map_ms_citation = json.load(f)

MODEL_NAME = "CSE_2023_MS_CitationMap"

In [7]:
# ==== Step 2. Rank-Normalize Citation Map ====
def convertUsingRankAndNormalize(map_open_alex_input):
  # Convert the dictionary to a pandas Series for easy manipulation
  citation_series = pd.Series(map_open_alex_input)

  # Filter out non-numeric values (like 0 for non-MS events) if necessary for ranking/normalization
  # Assuming you only want to rank/normalize MS events with citation counts > 0
  ms_citations = citation_series[citation_series.index.str.startswith('MS') & (citation_series > 0)]

  # Rank the citation counts
  # method='average' assigns the average rank to tied values
  # ascending=True ranks lowest values as 1st, highest as last
  ranked_citations = ms_citations.rank(method='average', ascending=True)

  # Normalize the ranked values to the range [0, 1]
  # Formula: (x - min(x)) / (max(x) - min(x))
  min_rank = ranked_citations.min()
  max_rank = ranked_citations.max()

  if max_rank == min_rank:
      # Avoid division by zero if all ranks are the same
      normalized_ranked_citations = ranked_citations - min_rank
  else:
      normalized_ranked_citations = (ranked_citations - min_rank) / (max_rank - min_rank)

  # Create a new map with the normalized ranked values,
  # keeping non-MS events or events with 0 citations as they were
  transformed_map = citation_series.copy()
  transformed_map[normalized_ranked_citations.index] = normalized_ranked_citations

  # Convert back to dictionary
  map_open_alex_all_ranked_normalized = transformed_map.to_dict()

  # Display some results to verify
  print("Original map sample:")
  display({k: map_open_alex_input[k] for k in list(map_open_alex_input)[:5] + list(map_open_alex_input)[-5:]})

  print("\nTransformed map sample:")
  display({k: map_open_alex_all_ranked_normalized[k] for k in list(map_open_alex_all_ranked_normalized)[:5] + list(map_open_alex_all_ranked_normalized)[-5:]})

  print("\nStatistics of normalized ranked values (only MS events > 0):")
  display(normalized_ranked_citations.describe())
  return map_open_alex_all_ranked_normalized

In [8]:
map_open_alex_all_ranked_normalized = convertUsingRankAndNormalize(map_open_alex_all)

Original map sample:


{'IP1': 0,
 'MS1': 9298.0,
 'MS2': 6627.0,
 'MS3': 1230.0,
 'MS4': 89.0,
 'MS418': 5065.0,
 'MS419': 1663.0,
 'MS420': 249.0,
 'MS421': 21310.0,
 'MS422': 2835.0}


Transformed map sample:


{'IP1': 0.0,
 'MS1': 0.6779661016949152,
 'MS2': 0.549636803874092,
 'MS3': 0.11380145278450363,
 'MS4': 0.012106537530266344,
 'MS418': 0.4406779661016949,
 'MS419': 0.15012106537530268,
 'MS420': 0.024213075060532687,
 'MS421': 0.9007263922518159,
 'MS422': 0.24213075060532688}


Statistics of normalized ranked values (only MS events > 0):


count    414.000000
mean       0.500000
std        0.289723
min        0.000000
25%        0.250000
50%        0.500000
75%        0.750000
max        1.000000
dtype: float64

In [9]:
print(map_open_alex_all_ranked_normalized)

{'IP1': 0.0, 'MS1': 0.6779661016949152, 'MS2': 0.549636803874092, 'MS3': 0.11380145278450363, 'MS4': 0.012106537530266344, 'MS5': 0.0387409200968523, 'MS6': 0.2711864406779661, 'MS7': 0.1234866828087167, 'MS10': 0.46973365617433416, 'MS11': 0.6900726392251816, 'MS12': 0.2009685230024213, 'MS13': 0.7651331719128329, 'MS14': 0.7772397094430993, 'MS15': 0.6295399515738499, 'MS16': 0.22033898305084745, 'MS17': 0.01694915254237288, 'MS18': 0.24939467312348668, 'MS19': 0.576271186440678, 'MS20': 0.3414043583535109, 'MS21': 0.22760290556900725, 'MS22': 0.7360774818401937, 'MS23': 0.3728813559322034, 'MS24': 0.07263922518159806, 'MS25': 0.2687651331719128, 'MS26': 0.801452784503632, 'MS27': 0.11864406779661017, 'MS28': 0.8401937046004843, 'MS29': 0.4915254237288136, 'MS30': 0.5980629539951574, 'MS31': 0.38498789346246975, 'MS32': 0.12106537530266344, 'MS33': 0.28329297820823246, 'MS34': 0.7966101694915254, 'MS35': 0.5786924939467313, 'MS84': 0.26150121065375304, 'MS373': 0.7554479418886199, 'I

In [10]:
prepareTimelineCitations3(data, map_open_alex_all_ranked_normalized, modelName = "CSE23 - OpenAlex Citations AVG (Ranked and Normalized 0-1)", fileName = "map_ms_citation_ranknorm.html")

→ map_ms_citation_ranknorm.html written with code only in tooltip and capped color scale.
